# レッスン6: 熱伝導方程式を解く — 第1部の卒業制作 🏗️🔥

レッスン1〜5の道具を全部つなげて、課題を完成させます。

> 厚さ300mmのコンクリート壁。初期温度0℃。t>0で両端が常に1℃。壁内部の温度分布の時間変動を計算し、3分後・30分後・2時間後・4時間後・8時間後・12時間後のグラフを描け。

---
## 6-1. 熱伝導方程式の直感

一次元非定常熱伝導方程式:

$$\frac{\partial T}{\partial t} = a \frac{\partial^2 T}{\partial x^2}$$

記号は怖そうですが、言っていることは1つだけ:

> **「ある点の温度は、自分が両隣の平均より低ければ上がり、高ければ下がる」**

- 左辺 ∂T/∂t … 温度の時間変化(どれだけ温まる/冷めるか)
- 右辺 ∂²T/∂x² … 「両隣の平均と自分の差」を表す量
- a … 熱拡散率。材料の「熱の伝わりやすさ」(レッスン1の1-Bで計算した約8.26×10⁻⁷ m²/s)

壁の端が1℃になれば、端の隣の点は「隣の平均より低い」ので温まる。それが順々に内側へ伝わる — 直感どおりの現象を式にしただけです。

---
## 6-2. 差分法 — 微分をコンピュータで扱える形にする

コンピュータは連続的な微分を直接は扱えないので、壁を**細かい点に区切り**、時間を**小さいステップに刻み**、微分を「差」で近似します。これが**差分法**(有限差分法)です。

数式を差分に置き換えて整理すると、更新式はこうなります:

$$T^{new}_i = T_i + r\,(T_{i-1} - 2T_i + T_{i+1}), \qquad r = \frac{a\,\Delta t}{\Delta x^2}$$

**レッスン4の練習4-Cでやった、あの1行そのもの**です。これを何千回も繰り返すだけ。

### 安定条件(レッスン3の3-Cの正体)

r が 0.5 を超えると計算が発散して壊れます(後で実験できます)。だから

$$\Delta t \le \frac{\Delta x^2}{2a}$$

を守って時間刻みを選びます。

---
## 6-3. 完成コード(お手本)

まず全体を実行して結果を見て、それから1行ずつ読み解きましょう。**どの行もレッスン1〜5でやったことしか使っていません。**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- 1. 物性値と条件の設定 (レッスン1: 変数) ----
L   = 0.3        # 壁の厚さ [m]
lam = 1.6        # 熱伝導率 λ [W/(m·K)]
rho = 2200       # 密度 ρ [kg/m³]
c   = 880        # 比熱 [J/(kg·K)]
a   = lam / (rho * c)          # 熱拡散率 [m²/s] (練習1-B)

# ---- 2. 空間と時間の分割 ----
n  = 30                        # 分割数
dx = L / n                     # 分割幅 [m]
dt = 20                        # 時間刻み [s]。安定条件 dx²/(2a)≈60s より小さい (練習3-C)
r  = a * dt / dx ** 2          # 更新式の係数
print(f"a = {a:.3e} m²/s,  r = {r:.3f}  (0.5以下なら安定)")

# ---- 3. 初期条件と境界条件 (レッスン4: 配列) ----
x = np.linspace(0, L, n + 1)   # 各点の位置 (練習4-A)
T = np.zeros(n + 1)            # 初期温度 全点0℃ (練習4-B)
T[0]  = 1.0                    # 両端は t>0 で常に1℃
T[-1] = 1.0

# ---- 4. グラフに描きたい時刻 [秒] (レッスン2: 辞書は次レッスンで学ぶのでリスト2本) ----
kiroku_jikoku = [3*60, 30*60, 2*3600, 4*3600, 8*3600, 12*3600]
kiroku_label  = ["3 min", "30 min", "2 h", "4 h", "8 h", "12 h"]

# ---- 5. 時間を進めるループ (レッスン2: for / レッスン3: if) ----
step_owari = int(12 * 3600 / dt)          # 12時間ぶんのステップ数
for step in range(1, step_owari + 1):
    T_new = T.copy()                                          # コピーを忘れない! (4-5)
    T_new[1:-1] = T[1:-1] + r * (T[:-2] - 2*T[1:-1] + T[2:])  # 更新式 (練習4-C)
    T = T_new
    # 端は書き換えていないので 1℃ のまま = 境界条件が保たれる

    t_now = step * dt                     # 現在時刻 [s]
    if t_now in kiroku_jikoku:            # 記録したい時刻なら線を1本描く
        label = kiroku_label[kiroku_jikoku.index(t_now)]
        plt.plot(x * 1000, T, marker="o", markersize=3, label=label)

# ---- 6. グラフの仕上げ (レッスン5) ----
plt.xlabel("x [mm]")
plt.ylabel("T [C]")
plt.title("1D transient heat conduction in a concrete wall")
plt.legend()
plt.grid(True)
plt.ylim(0, 1.05)
plt.show()

### 結果の読み方(物理の考察もレポートに書けるように)

- **3分後**: 端のごく近くだけ温まり、内部はまだ0℃のまま
- **30分〜2時間**: 熱がじわじわ内側へ。中央はまだ冷たい
- **4〜8時間**: 中央にも熱が届き、分布がU字→浅いU字に
- **12時間後**: 全体がほぼ1℃に近づく(最終的には全点1℃の定常状態へ)

コンクリート300mmの壁は「外気温の変化が中まで届くのに半日かかる」— 建物の壁が厚いと室温が安定する理由が、自分の計算で確認できたことになります。

---
## 6-4. 自分で組み立て直す(ここが本番)

お手本を見ながらでいいので、**新しいセルに自分の手で全部打ち直してください**(写経)。コピペ禁止。打ちながら「この行はレッスン◯のあれだ」と思い出すのが目的です。実務でも新しい技術はまず写経から入ることが多いです。

In [ ]:
# ここに自分の手で組み立て直す


---
## ✏️ 実験課題(理解を深める)

写経が終わったら、パラメータをいじって「なぜ?」を考えてみてください:

**6-A** 壁厚を600mmにすると12時間後の中央温度はどうなる?(予想してから実行!)

**6-B** `dt = 100` に変えると?(r を計算してから予想。r > 0.5 になるように dt を大きくしていくと、グラフがギザギザに発散する「不安定」が観察できます。数値計算が壊れる瞬間を一度自分の目で見ておくのは超大事)

**6-C** 「両端」ではなく「左端だけ1℃(右端は0℃のまま)」にすると分布はどうなる?

---
## 🎓 第1部 修了!

ここまでで身についたもの:

| 概念 | 実務での使われ方 |
|------|------------------|
| 変数・計算 | すべてのコードの土台 |
| リスト・ループ・条件分岐 | すべての言語に共通する制御の骨格 |
| 関数 | コードの部品化。実務コードの基本単位 |
| numpy / matplotlib | データ処理・science系実務の標準 |
| 「初期化→ループで更新→出力」 | シミュレーション・バッチ処理・集計処理に共通する**プログラムの典型構造** |

**次 → 第2部(レッスン7〜)**: 辞書・例外処理・ファイル・クラスなど、実務コードを読み書きするための残りの文法へ。